# Lesson 03 Lab — PyTorch AMP: autocast and GradScaler

**Puzzle:** Can mixed-precision training be reduced to wrapping the forward pass in autocast?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Mixed-precision training is a feedback system. Autocast chooses operation dtypes during the forward pass, gradient scaling changes the numerical range seen by backward, and the optimizer must only step after gradients have been checked and unscaled. Demonstrating one BF16 activation therefore proves much less than demonstrating a complete, finite parameter update.


## 0. Predict before running

1. Predict the dtype of model parameters and forward outputs inside BF16 autocast.
2. Predict whether GradScaler's scale should change when every gradient stays finite.
3. Name the observation that proves an optimizer update occurred rather than only a forward pass.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

The AMP loop contains FP32 parameters and optimizer state, autocast-selected forward activations, gradients, a scalar loss scale, and an optimizer update. These objects do not all share one dtype or lifetime.

- Autocast selects lower precision per eligible operation; it does not permanently convert every tensor.
- GradScaler changes loss magnitude before backward, unscales gradients before the optimizer step, and adapts its scale.
- The optimizer state and usually the master parameters remain higher precision.


## 2. Derive the mechanism

If `g` is the true gradient and `S` is the loss scale, backward first produces `S·g`; unscale restores `g` before clipping or the optimizer step. `GradScaler` skips the step when non-finite gradients are found and adapts `S`. Autocast independently chooses eligible forward-operation dtypes.

Let the unscaled loss be `L` and the current scale be `S`. Backward differentiates `S·L`, producing scaled gradients `S·g`. Before the optimizer step, GradScaler divides by S and checks for Inf/NaN. If the check passes, the optimizer consumes g; if it fails, the step is skipped and the scale policy reacts. The ordering is semantic: clipping or inspecting gradients before unscale changes their meaning.

Autocast is a dispatch policy, not a recursive call to `.to(bfloat16)` on the entire model. Eligible compute-heavy operations may emit BF16 while parameters and optimizer state remain FP32. BF16 does not usually need scaling for range in the way FP16 does, but exercising the full scaler API is still useful because the lesson is about the control loop and its evidence, not a single recommended dtype recipe.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "03-pytorch-amp"
device = require_cuda()
torch.manual_seed(2026 + 3)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | FP32 parameters and optimizer state outside autocast |
| Candidate | BF16 autocast forward wrapped in a complete scale/backward/step/update loop |
| Held constant | same MLP, batch, targets, optimizer, seed, and six training steps |
| Measurements | loss history, output dtype, parameter dtype, gradient finiteness, scaler value |
| Evidence | `pytorch-gpu` |

**Experiment:** Train a small CUDA MLP with BF16 autocast and GradScaler while recording loss, parameter dtype, output dtype, gradient finiteness, and scale history.


## 5. Read the experiment code

The notebook prints parameter and output dtypes, runs the complete update loop, and records gradient finiteness rather than stopping after one autocast forward.

The environment cell verifies CUDA and fixes the random seed. The experiment constructs one small MLP, keeps its parameters in FP32, enters the autocast context only for forward and loss computation, and then executes the scaler sequence. Every step records five pieces of state so the notebook can distinguish dispatch, numerical health, and optimization progress.

A decreasing toy loss is not a model-quality claim; it is a control-flow check. The stronger invariants are that every recorded gradient is finite, the output is BF16 under autocast, parameters remain FP32, and the loop reaches optimizer updates without an error output.

Only after these variables match the protocol should the cell be executed.


In [2]:
model = torch.nn.Sequential(torch.nn.Linear(512, 1024), torch.nn.GELU(), torch.nn.Linear(1024, 64)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler("cuda")
x = torch.randn(256, 512, device=device); target = torch.randn(256, 64, device=device)
history = []
for step in range(6):
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        pred = model(x); loss = torch.nn.functional.mse_loss(pred, target)
    scaler.scale(loss).backward()
    finite = all(p.grad is None or torch.isfinite(p.grad).all().item() for p in model.parameters())
    scale_before = scaler.get_scale(); scaler.step(optimizer); scaler.update()
    history.append({"step": step, "loss": round(loss.item(), 7), "output_dtype": str(pred.dtype),
                    "grads_finite": bool(finite), "scale": float(scale_before)})
result = base_result(3, "pytorch-gpu")
result.update({"parameter_dtype": str(next(model.parameters()).dtype), "history": history,
               "conclusion": "The full autocast-scale-backward-step-update loop completed with finite gradients."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Initial loss | 1.037629 |
| Final loss | 0.596515 |
| Autocast output dtype | torch.bfloat16 |
| All recorded gradients finite | yes |
| Parameter dtype | torch.float32 |
| Final scaler value | 65536.000000 |


## 7. Interpret rather than merely print

The six saved steps reduced loss from 1.0376294 to 0.5965154. Every output was `torch.bfloat16`, every gradient check returned true, and parameters remained `torch.float32`. The scale stayed at 65536 because no non-finite event forced the policy to back off during this short run.

Taken together, those fields establish a functioning mixed-precision loop on this PyTorch/CUDA stack. They do not establish faster training, convergence parity on a real dataset, or the best scale-growth policy. Those require longer runs with repeated timing and a frozen quality target.

**Inspection rule:** A valid loop needs finite gradients and an optimizer update. An autocast dtype printout alone is not a training result.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "The full autocast-scale-backward-step-update loop completed with finite gradients.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:11+00:00",
  "history": [
    {
      "grads_finite": true,
      "loss": 1.0376294,
      "output_dtype": "torch.bfloat16",
      "scale": 65536.0,
      "step": 0
    },
    {
      "grads_finite": true,
      "loss": 0.9307048,
      "output_dtype": "torch.bfloat16",
      "scale": 65536.0,
      "step": 1
    },
    {
      "grads_finite": true,
      "loss": 0.836614,
      "output_dtype": "torch.bfloat16",
      "scale": 65536.0,
      "step": 2
    },
    {
      "grads_finite": true,
      "loss": 0.751017,
      "output_dtype": "torch.bfloat16",
      "scale": 65536.0,
      "step": 3
    },
    {
      "

## 9. Make the bounded decision

> AMP is a control loop across forward, backward, unscale, step, and update—not a global dtype switch.

**Acceptance/rollback:** Verify the order `zero_grad -> autocast forward -> scale(loss).backward -> unscale/step -> update`, record finite gradients and scale history, and keep the loss objective identical to the FP32 baseline.

**Failure analysis:** Common failures include calling `optimizer.step()` directly on scaled gradients, clipping before `unscale_`, moving master parameters to FP16, or judging success only from the forward dtype. A finite loss can coexist with zeroed small gradients, and a skipped optimizer step can be invisible unless the scale and parameter update are inspected.


## 10. Extend the evidence

Add a deliberately overflowing step and verify that GradScaler skips the update and changes its scale. Then time FP32, FP16+scaler, and BF16 autocast over a longer MLP while comparing the same validation loss trajectory. Preserve the exact optimizer, seed, and batch order so numerical and throughput decisions are not confounded.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
